In [2]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.normalize as N, src.keys as KY, src.blocking as B, src.rerank as R, src.features as F, src.metrics as M

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
VALID_GT = ROOT / "notebooks" / "phase2_valid_gt.parquet"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="4GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)

S1_ALL    = k2("train_source1")
TGT_PARTS = [k2("train_source2"), k2("train_source3")]
TGT       = "(" + " UNION ALL ".join(f"SELECT * FROM {p}" for p in TGT_PARTS) + ")"
ALL_SRC   = [f"{s}_source{i}" for s in ("train", "test") for i in (1, 2, 3)]

print(sys.executable)
print(con.execute("SELECT (SELECT count(*) FROM eval_s1), (SELECT count(*) FROM eval_pairs), "
                  "(SELECT count(*) FROM train_s1), (SELECT count(*) FROM train_pairs)").fetchone())

D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Scripts\python.exe
(20000, 69252, 30000, 104127)


In [3]:
for s in ["Maison Comice et Fils", "Maison Comice & Fils", "SARL Maison Comice", "Union de Gitane France",
          "Ste Generale SA", "Société Générale", "Ets Martin Freres", "Etablissements Martin",
          "Sri Shiv Solutions Pvt Ltd", "clyialsystems.com", "Dynamics Photonics L L C"]:
    v = N.normalize_name_v2(s)
    print(f"{s!r:32} -> {v!r:36} core={N.name_core(v)!r}")
for a in ["N° 14 rue François Joseph Nollet, Lille", "14 rue Francois Joseph Nollet Lille",
          "Plot No 21, Chandan Place", "03211 Driftwood Lane"]:
    print(f"{a!r:44} -> {N.normalize_address_v2(a)!r}")
print("typo channel present:", "'p6'" in B.keys_sql("x", "y", "us"))

'Maison Comice et Fils'          -> 'maison comice and fils'             core='maison comice'
'Maison Comice & Fils'           -> 'maison comice and fils'             core='maison comice'
'SARL Maison Comice'             -> 'sarl maison comice'                 core='sarl maison comice'
'Union de Gitane France'         -> 'union de gitane france'             core='union de gitane france'
'Ste Generale SA'                -> 'societe generale sa'                core='societe generale'
'Société Générale'               -> 'societe generale'                   core='societe generale'
'Ets Martin Freres'              -> 'etablissements martin freres'       core='etablissements martin'
'Etablissements Martin'          -> 'etablissements martin'              core='etablissements martin'
'Sri Shiv Solutions Pvt Ltd'     -> 'sri shiva solutions private limited' core='shiva solutions'
'clyialsystems.com'              -> 'clyialsystems'                      core='clyialsystems'
'Dynamics Photonics L

In [4]:
t0 = time.time()
KY.build_all(con, PARQ, KEYS2, ALL_SRC, overwrite=True)
print(f"keys rebuilt in {(time.time() - t0) / 60:.1f} min")

B.run_blocking(con, "cand2", S1_ALL, TGT_PARTS, s1_ids_tbl="eval_s1", k_max=50)
display(pd.concat([B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"cycle2_top{k}/src")
                   for k in (5, 10, 20, 30, 50)], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source1      146.6s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source2      324.1s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 india    target keys      243s
   cand2 india    key weights      243s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,cycle2_top5/src,0.952001,0.996778,0.985681,9.95595,10.0,10.0,10
1,cycle2_top10/src,0.966153,0.997464,0.989390,19.75435,20.0,20.0,20
2,cycle2_top20/src,0.973084,0.998151,0.991714,38.77590,40.0,40.0,40
3,cycle2_top30/src,0.975986,0.998415,0.992611,57.00195,60.0,60.0,60
4,cycle2_top50/src,0.979235,0.998679,0.993658,91.13675,100.0,100.0,100


In [5]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

B.run_blocking(con, "cand_tr", S1_ALL, TGT_PARTS, s1_ids_tbl="train_s1", k_max=50)

t0 = time.time()
tr_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({R.feat_sql('cand_tr', S1_ALL, TGT)}) f LEFT JOIN train_pairs p USING (s1_id, tgt_id)""").fetchdf()
ev_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({R.feat_sql('cand2', S1_ALL, TGT)}) f LEFT JOIN eval_pairs p USING (s1_id, tgt_id)""").fetchdf()
print(f"train {len(tr_df):,} | eval {len(ev_df):,}  ({time.time() - t0:.0f}s)")

rr = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
rr.fit(tr_df[R.FEATS], tr_df["y"])
rr.booster_.save_model(str(ARTIFACTS / "rerank_lgb_c2.txt"))
ev_df["p"] = rr.predict_proba(ev_df[R.FEATS])[:, 1]
print(f"eval AUC: {roc_auc_score(ev_df.y, ev_df.p):.4f}")

con.register("ev_scored", ev_df[["s1_id", "tgt_id", "src", "p"]])
con.execute("""CREATE OR REPLACE TABLE cand_rr AS
    SELECT *, row_number() OVER (PARTITION BY s1_id, src ORDER BY p DESC, tgt_id) AS rk2 FROM ev_scored""")

rows = [B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand_rr WHERE rk2 <= {cap} AND p >= {tau})",
                        f"p>={tau} cap{cap}")
        for tau in (0.003, 0.01, 0.03) for cap in (5, 8, 10)]
display(pd.concat(rows, ignore_index=True))

del tr_df; gc.collect()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand_tr india    token df          10s
   cand_tr india    S1 keys           12s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train 2,724,802 | eval 1,822,735  (52s)
eval AUC: 0.9990


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,p>=0.003 cap5,0.974817,0.998574,0.992619,6.15650,6.0,10.0,10
1,p>=0.003 cap8,0.976607,0.998679,0.993011,6.60530,6.0,16.0,16
2,p>=0.003 cap10,0.976737,0.998679,0.993045,6.67960,6.0,17.0,20
3,p>=0.01 cap5,0.973185,0.998468,0.992114,5.47820,5.0,10.0,10
4,p>=0.01 cap8,0.974586,0.998521,0.992380,5.67190,5.0,13.0,16
5,p>=0.01 cap10,0.974614,0.998521,0.992385,5.68695,5.0,13.0,20
6,p>=0.03 cap5,0.969156,0.998151,0.990837,4.75790,5.0,9.0,10
7,p>=0.03 cap8,0.970037,0.998151,0.990967,4.81235,5.0,10.0,16
8,p>=0.03 cap10,0.970051,0.998151,0.990969,4.81430,5.0,10.0,20


131

In [6]:
del ev_df; gc.collect()
importlib.reload(B); importlib.reload(R); importlib.reload(F)
con.execute("PRAGMA threads=8")
con.execute("SET memory_limit='5GB'")

TAU_CUT, CAP_CUT, K_MAX = 0.003, 8, 50
reranker = R.load(ARTIFACTS / "rerank_lgb_c2.txt")

con.execute("CREATE OR REPLACE TABLE cut_full (s1_id BIGINT, tgt_id BIGINT, src BIGINT, p DOUBLE, rk2 BIGINT)")
t0 = time.time()
B.run_blocking(con, "cut_full", S1_ALL, TGT_PARTS, s1_ids_tbl=None, k_max=K_MAX, s1_chunks=64,
               on_chunk=lambda tbl: R.rerank_and_cut(con, tbl, S1_ALL, TGT, reranker, "cut_full",
                                                     tau=TAU_CUT, cap=CAP_CUT))
print(f"done in {(time.time() - t0) / 60:.1f} min")

print(con.execute("SELECT count(*), count(DISTINCT s1_id), count(*) / count(DISTINCT s1_id) FROM cut_full").fetchone())
display(B.eval_blocking(con, "(SELECT s1_id, tgt_id FROM cut_full)", "cycle2 full cut, eval S1"))

F.build_rev(con, "cut_full", "cut_rev")
print("cut_rev rows:", con.execute("SELECT count(*) FROM cut_rev").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cut_full india    token df           6s
   cut_full india    S1 keys           54s
   cut_full india    target keys      136s
   cut_full india    key weights      139s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

   cut_full us       scored 64/64    8287s
done in 138.2 min
(14576592, 2206483, 6.606256200478318)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,"cycle2 full cut, eval S1",0.976578,0.998679,0.993015,6.6043,6.0,16.0,16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cut_rev rows: 14576592


In [8]:
import sys, numpy as np, xgboost as xgb, time
print(sys.executable, xgb.__version__)
X = np.random.rand(200_000, 40).astype("float32"); y = (X[:, 0] + X[:, 1] > 1).astype(int)
t0 = time.time(); xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=100).fit(X, y)
print(f"GPU OK {time.time() - t0:.1f}s")

D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Scripts\python.exe 3.4.1
GPU OK 1.3s


In [9]:
import xgboost as xgb, lightgbm as lgb
from sklearn.metrics import roc_auc_score
importlib.reload(F); F.setup_macros(con)
con.execute("PRAGMA threads=8"); con.execute("SET memory_limit='5GB'")

# token IDF over the train target pool
F.build_name_idf(con, TGT)

# labels for the full train pool
con.execute(f"""
CREATE OR REPLACE TABLE gt_all AS
SELECT eid(source1_entity_id) AS s1_id, eid(trim(tgt)) AS tgt_id
FROM (SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS tgt
      FROM {pq('train_ground_truth')}
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")

# 600k training S1: not validation, not the reranker's training sample
con.execute(f"""
CREATE OR REPLACE TABLE m_tr_s1 AS
SELECT s1_id FROM (
    SELECT DISTINCT s1_id FROM cut_rev
    ANTI JOIN (SELECT eid(source1_entity_id) AS s1_id FROM read_parquet('{VALID_GT.as_posix()}')) v USING (s1_id)
    ANTI JOIN train_s1 USING (s1_id)
) USING SAMPLE 600000 ROWS (reservoir, 21)
""")
con.execute("""CREATE OR REPLACE TABLE m_tr_rows AS
    SELECT r.*, (g.tgt_id IS NOT NULL)::INT AS y
    FROM cut_rev r SEMI JOIN m_tr_s1 USING (s1_id) LEFT JOIN gt_all g USING (s1_id, tgt_id)""")
con.execute("""CREATE OR REPLACE TABLE m_ev_rows AS
    SELECT r.*, (g.tgt_id IS NOT NULL)::INT AS y
    FROM cut_rev r SEMI JOIN eval_s1 USING (s1_id) LEFT JOIN eval_pairs g USING (s1_id, tgt_id)""")
print(con.execute("SELECT (SELECT count(*) FROM m_tr_rows), (SELECT count(*) FROM m_ev_rows)").fetchone())

def feats_with_labels(rows_tbl, n_chunks):
    df = F.build_features_chunked(con, rows_tbl, S1_ALL, TGT, n_chunks=n_chunks)
    lab = con.execute(f"SELECT s1_id, tgt_id, y FROM {rows_tbl}").fetchdf()
    return df.merge(lab, on=["s1_id", "tgt_id"])

t0 = time.time()
tr = feats_with_labels("m_tr_rows", 24)
ev = feats_with_labels("m_ev_rows", 2)
print(f"train {tr.shape} pos={tr.y.mean():.3f} | eval {ev.shape}  ({(time.time() - t0) / 60:.1f} min)")

# early-stopping split by S1
rng = np.random.default_rng(0)
es = tr["s1_id"].isin(set(rng.choice(tr["s1_id"].unique(), size=tr["s1_id"].nunique() // 10, replace=False)))

t0 = time.time()
mx = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=4000, learning_rate=0.03,
                       max_depth=10, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                       eval_metric="logloss", early_stopping_rounds=150)
mx.fit(tr.loc[~es, F.FEATS_V2], tr.loc[~es, "y"],
       eval_set=[(tr.loc[es, F.FEATS_V2], tr.loc[es, "y"])], verbose=500)
mx.save_model(str(ARTIFACTS / "matcher_xgb_v2.json"))
print(f"trained in {(time.time() - t0) / 60:.1f} min, best iter {mx.best_iteration}")

ev["q_v2"] = mx.predict_proba(ev[F.FEATS_V2])[:, 1]
ev["q_v1"] = lgb.Booster(model_file=str(ARTIFACTS / "matcher_lgb.txt")).predict(ev[F.FEATS])
print(f"eval AUC  v1: {roc_auc_score(ev.y, ev.q_v1):.4f} | v2: {roc_auc_score(ev.y, ev.q_v2):.4f}")

con.register("ev_q", ev[["s1_id", "tgt_id", "q_v1", "q_v2"]])
rows = []
for col in ("q_v1", "q_v2"):
    for t in (0.5, 0.6, 0.7, 0.8):
        rows.append(M.eval_f05(con, f"(SELECT s1_id, tgt_id FROM ev_q WHERE {col} >= {t})", f"{col} >= {t}"))
display(pd.concat(rows, ignore_index=True))
display(pd.Series(mx.feature_importances_, index=F.FEATS_V2).sort_values(ascending=False).head(20).to_frame("imp").T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3962985, 132086)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

train (3962985, 52) pos=0.512 | eval (132086, 52)  (7.0 min)
[0]	validation_0-logloss:0.66643
[500]	validation_0-logloss:0.04663
[1000]	validation_0-logloss:0.04317
[1500]	validation_0-logloss:0.04224
[2000]	validation_0-logloss:0.04186
[2500]	validation_0-logloss:0.04171
[2910]	validation_0-logloss:0.04170
trained in 2.6 min, best iter 2760


D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [15:13:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


eval AUC  v1: 0.9979 | v2: 0.9989


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q_v1 >= 0.5,0.965888,0.915809,0.968715,0.981228,0.953315,3.36410
1,q_v1 >= 0.6,0.968074,0.936389,0.969863,0.985841,0.948088,3.33000
2,q_v1 >= 0.7,0.969602,0.953227,0.970526,0.990163,0.941864,3.29370
3,q_v1 >= 0.8,0.968603,0.969130,0.968573,0.993360,0.933244,3.25305
4,q_v2 >= 0.5,0.974313,0.951356,0.975610,0.986930,0.958427,3.36260
5,q_v2 >= 0.6,0.975857,0.962582,0.976607,0.990702,0.953936,3.33410
6,q_v2 >= 0.7,0.976537,0.971001,0.976850,0.993578,0.949503,3.30900
7,q_v2 >= 0.8,0.975674,0.980355,0.975410,0.995944,0.943236,3.27935


,p_margin,p,legal_cmp,first_tok_eq,p_gap_own,t_addr_null,tgt_max_p,xs_addr_jacc,idf_unshared_max,name_n_unshared,cmp_contains,name_jacc,own_n_hi,t_nums,other_best_p,first_num_cmp,wratio,addr_tset,addr_alpha_jacc,cmp_prefix
imp,0.446361,0.208505,0.037157,0.024125,0.018765,0.015433,0.015148,0.014702,0.013448,0.012684,0.012028,0.011751,0.011367,0.010359,0.009973,0.008516,0.008145,0.007702,0.006902,0.006411


In [10]:
TAU = 0.7
ev["pred"] = ev["q_v2"] >= TAU
tp_, fn_, fp_ = ev[(ev.y == 1) & ev.pred], ev[(ev.y == 1) & ~ev.pred], ev[(ev.y == 0) & ev.pred]
not_in_cands = con.execute("SELECT count(*) FROM eval_pairs ANTI JOIN m_ev_rows USING (s1_id, tgt_id)").fetchone()[0]
print(f"true pairs {int(ev.y.sum()) + not_in_cands:,} | found {len(tp_):,} | missed in candidates {not_in_cands:,} "
      f"| rejected by matcher {len(fn_):,} | false matches {len(fp_):,}")

# 1. what do errors have in common? (rate of each property among TP / FN / FP)
def profile(d):
    return {
        "n": len(d),
        "tgt addr missing":        d.t_addr_null.mean(),
        "name_jw < 0.7":           (d.name_jw < 0.7).mean(),
        "addr_jacc < 0.3":         (d.addr_jacc.fillna(0) < 0.3).mean(),
        "house number conflict":   (d.first_num_cmp == 0).mean(),
        "legal form conflict":     (d.legal_cmp == 0).mean(),
        "contested (n_claim > 1)": (d.n_claim > 1).mean(),
        "not top claimant":        (d.rev_rk > 1).mean(),
        "source S3":               (d.src == 3).mean(),
        "reranker p < 0.5":        (d.p < 0.5).mean(),
    }
display(pd.DataFrame({"TP": profile(tp_), "FN (missed)": profile(fn_), "FP (false)": profile(fp_)}).round(3))

# 2. what do they look like? (raw + normalized text; for FPs, who really owns the target)
RAW_T = f"(SELECT * FROM {pq('train_source2')} UNION ALL SELECT * FROM {pq('train_source3')})"

def show(d, n=25):
    s = d.sample(min(n, len(d)), random_state=0)[["s1_id", "tgt_id", "q_v2", "p", "rev_rk", "n_claim"]]
    con.register("_s", s)
    out = con.execute(f"""
    SELECT round(_s.q_v2, 3) AS q, round(_s.p, 3) AS p, _s.rev_rk, _s.n_claim,
           a.country_key AS cty, a.name_v2 AS s1_name, b.name_v2 AS tgt_name,
           r.business_name AS tgt_raw_name, a.addr_v2 AS s1_addr, b.addr_v2 AS tgt_addr,
           o.name_v2 AS true_owner_name
    FROM _s
    JOIN (SELECT eid(entity_id) AS id, * FROM {S1_ALL}) a ON a.id = _s.s1_id
    JOIN (SELECT eid(entity_id) AS id, * FROM {TGT})    b ON b.id = _s.tgt_id
    JOIN (SELECT eid(entity_id) AS id, business_name FROM {RAW_T}) r ON r.id = _s.tgt_id
    LEFT JOIN gt_all g ON g.tgt_id = _s.tgt_id
    LEFT JOIN (SELECT eid(entity_id) AS id, name_v2 FROM {S1_ALL}) o ON o.id = g.s1_id
    ORDER BY q DESC""").fetchdf()
    con.unregister("_s")
    return out

pd.set_option("display.max_colwidth", 45)
print("=== FALSE NEGATIVES (true match, rejected) ==="); display(show(fn_))
print("=== FALSE POSITIVES (predicted, wrong) ===");     display(show(fp_))

true pairs 69,252 | found 65,755 | missed in candidates 1,622 | rejected by matcher 1,875 | false matches 425


,TP,FN (missed),FP (false)
n,65755.000,1875.000,425.000
tgt addr missing,0.021,0.418,0.125
name_jw < 0.7,0.057,0.099,0.115
addr_jacc < 0.3,0.040,0.468,0.214
house number conflict,0.137,0.449,0.546
legal form conflict,0.003,0.013,0.012
contested (n_claim > 1),0.264,0.586,0.327
not top claimant,0.003,0.339,0.045
source S3,0.515,0.518,0.527
reranker p < 0.5,0.049,0.731,0.367


=== FALSE NEGATIVES (true match, rejected) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,q,p,rev_rk,n_claim,cty,s1_name,tgt_name,tgt_raw_name,s1_addr,tgt_addr,true_owner_name
0,0.697,0.787,2.0,2.0,india,lilawati and sons private limited,lilawati and sons private limited,Lilawati & Sons Private-Ltd,444 bazar bank street main road chorda ch...,NaN,lilawati and sons private limited
1,0.695,0.584,1.0,1.0,us,environmental center,environmental center,Environmental Center,2731 barrington road unit unit a tontitow...,273 barrington road unit a springddale ar,environmental center
2,0.691,0.579,1.0,2.0,us,patriot fellowship,patriot fellowship,patriot fellowship,489 klein road amherst ny,488 klein road buffalo city ny,patriot fellowship
3,0.678,0.837,1.0,3.0,india,sainath india india private limited,sainath india india private,Sainath (India) India Pvt,35 n 3 4 first floor murti bhawan rampur ...,5 n 3 4 first floor murti bhawan rampur g...,sainath india india private limited
4,0.604,0.551,1.0,2.0,india,jsn management private limited,jsn management private,Jsn Management Pvt.,107 harmony plaza agrawal nagar sneh naga...,NaN,jsn management private limited
5,0.587,0.344,1.0,1.0,us,leoine emanuel creative spark incorporated,leoine emanuel creative spark,Leoine Emanuel Creative Spark,wa spokane valley 13724 everett avenue,13725 everett avenue spokane valley wa,leoine emanuel creative spark incorporated
6,0.514,0.282,1.0,1.0,us,empire fellowship,empire felloslwshmp,EMPIRE FELLOSLWSHMP,ca unit 304 ventura 6700 telephone road,700 telephone road po box 8427 ventura cd...,empire fellowship
7,0.492,0.337,3.0,4.0,india,natural technologies private limited,mr natural technologies private,Mr Natural Technologies Private,1 1b 1 gandhi road avinashi main road anu...,NaN,natural technologies private limited
8,0.492,0.281,1.0,1.0,india,tech exports private limited,tech exports private limited,Tech Exports Pvt Ltd.,141 4 chairmannallamutthu pillai road mah...,143 4 chairmannallamutthu pillai road mah...,tech exports private limited
9,0.417,0.011,3.0,3.0,india,raj energy,raj services,Raj Sérvices,plot no 25 red cross road sadar nagpur ma...,no 25 nagpur maharashtra,raj energy


=== FALSE POSITIVES (predicted, wrong) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,q,p,rev_rk,n_claim,cty,s1_name,tgt_name,tgt_raw_name,s1_addr,tgt_addr,true_owner_name
0,0.997,1.000,1.0,1.0,us,regional investments networks pc,regional investments investments networks...,Regional Investment Investment Networks Llc,6695 street route 15 unit 22 bryan oh,6695 street route 15 unit 22 bryan oh,NaN
1,0.990,0.990,1.0,6.0,india,shree foundation private limited,shree foundation,Shree Foundation,gat no 388 1 devendra residency bangla va...,gat no 388 mahalunge tal khed pune mahara...,NaN
2,0.987,0.769,1.0,2.0,india,accord seva samiti,accord seva samiti corporation,Accord Seva Samiti [Corp],floor c 301 kunal puram s no 392 393 bomb...,NaN,NaN
3,0.978,0.004,1.0,1.0,india,ace galaxy consultancy private limited,zephyuma 5961,Zephyuma (#5961),2 a first street north boag road t nagar ...,2 a first street north boag road t nagar ...,punit foods private limited
4,0.975,0.075,1.0,1.0,india,mex doors,cdfsolutions,CDFSOLUTIONS.COM,shop 1 & 2 jubileeview yari road andheri ...,maharashtra door no 71 shop 1 & 2 jubilee...,NaN
5,0.972,0.681,1.0,2.0,india,shx club private limited,shx shx club limited,SHX SHX Club Limited,bb 9 greater kailash enclave ii new delhi...,bb 16 greater kailash enclave ii delhi,NaN
6,0.951,0.385,1.0,1.0,us,ace genetics incorporated,ace genetics incorporated,Ace Génetics Inc,16301 pacific avenue independence mo,16305 pacific avenue independence mo,NaN
7,0.936,0.882,1.0,1.0,us,montano clear hall,montano clear hall limited,Montano Cléar HALL LTD,1100 1480 alvord tx floor 0,1102 1480 floor 0 alvord tx,NaN
8,0.936,0.340,1.0,1.0,us,environmental center,environmental center company,Environmental Center Co,2731 barrington road unit unit a tontitow...,springdale 2735 barrington road ar,NaN
9,0.888,0.788,1.0,1.0,india,durable and company,durable and company llp,Durable & Co L.L.P.,1st block karnataka no 217 8 c main banga...,no 229 8 c main bengaluru karnataka 1st b...,NaN


In [11]:
importlib.reload(F); F.setup_macros(con)
del tr; gc.collect()

def feats_with_labels(rows_tbl, n_chunks, feats):
    df = F.build_features_chunked(con, rows_tbl, S1_ALL, TGT, n_chunks=n_chunks, feats=feats)
    lab = con.execute(f"SELECT s1_id, tgt_id, y FROM {rows_tbl}").fetchdf()
    return df.merge(lab, on=["s1_id", "tgt_id"])

t0 = time.time()
tr  = feats_with_labels("m_tr_rows", 24, F.FEATS_V3)
ev3 = feats_with_labels("m_ev_rows", 2, F.FEATS_V3)
print(f"train {tr.shape} | eval {ev3.shape}  ({(time.time() - t0) / 60:.1f} min)")

es = tr["s1_id"].isin(set(rng.choice(tr["s1_id"].unique(), size=tr["s1_id"].nunique() // 10, replace=False)))
mx3 = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=5000, learning_rate=0.03,
                        max_depth=10, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", early_stopping_rounds=150)
mx3.fit(tr.loc[~es, F.FEATS_V3], tr.loc[~es, "y"],
        eval_set=[(tr.loc[es, F.FEATS_V3], tr.loc[es, "y"])], verbose=1000)
mx3.save_model(str(ARTIFACTS / "matcher_xgb_v3.json"))

ev3 = ev3.merge(ev[["s1_id", "tgt_id", "q_v2"]], on=["s1_id", "tgt_id"])
ev3["q_v3"] = mx3.predict_proba(ev3[F.FEATS_V3])[:, 1]
print(f"eval AUC  v2: {roc_auc_score(ev3.y, ev3.q_v2):.4f} | v3: {roc_auc_score(ev3.y, ev3.q_v3):.4f}")

con.register("ev_q3", ev3[["s1_id", "tgt_id", "q_v2", "q_v3"]])
rows = [M.eval_f05(con, f"(SELECT s1_id, tgt_id FROM ev_q3 WHERE {c} >= {t})", f"{c} >= {t}")
        for c in ("q_v2", "q_v3") for t in (0.6, 0.7, 0.8)]
display(pd.concat(rows, ignore_index=True))
display(pd.Series(mx3.feature_importances_, index=F.FEATS_V3).sort_values(ascending=False).head(15).to_frame("imp").T)

# how many remaining misses are name-only and contested (likely unrecoverable)?
fn3 = ev3[(ev3.y == 1) & (ev3.q_v3 < 0.7)]
print(f"FN {len(fn3):,} | name-only + contested: {((fn3.t_addr_null == 1) & (fn3.n_claim > 1)).mean():.2f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   features 4/24      73s


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (3962985, 61) | eval (132086, 61)  (7.7 min)
[0]	validation_0-logloss:0.66610
[1000]	validation_0-logloss:0.03524
[2000]	validation_0-logloss:0.03420
[2984]	validation_0-logloss:0.03413
eval AUC  v2: 0.9989 | v3: 0.9992


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q_v2 >= 0.6,0.975857,0.962582,0.976607,0.990702,0.953936,3.33410
1,q_v2 >= 0.7,0.976537,0.971001,0.976850,0.993578,0.949503,3.30900
2,q_v2 >= 0.8,0.975674,0.980355,0.975410,0.995944,0.943236,3.27935
3,q_v3 >= 0.6,0.980355,0.971936,0.980831,0.993123,0.959265,3.34455
4,q_v3 >= 0.7,0.980848,0.975678,0.981140,0.995115,0.956088,3.32680
5,q_v3 >= 0.8,0.980312,0.979420,0.980362,0.996570,0.952362,3.30900


,p_margin,p,xs_num_eq,num_trunc,legal_cmp,num_absdiff_log,first_tok_eq,p_gap_own,first_num_cmp,num_lev,tgt_max_p,t_nums,own_n_hi,idf_unshared_max,t_addr_null
imp,0.382173,0.186939,0.080717,0.039337,0.01941,0.018162,0.017462,0.017029,0.016034,0.012783,0.01277,0.012423,0.010495,0.010389,0.010121


FN 1,419 | name-only + contested: 0.54
